<a href="https://colab.research.google.com/github/aarnav-y/HCL_Project/blob/main/employee_attendance_analysis_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Employee Attendance Analysis

**Project objective:** Study employee attendance, department and role distribution, tenure, satisfaction, overtime and attrition using data stored in a CSV file.


**Python concepts used:** variables, data types, lists, dictionaries, operators, `if/elif/else`, `for` loops, functions, string formatting and file handling.



## 1. Create the CSV file automatically

This cell writes 60 sample employee records to a CSV file.

In [1]:
import csv

file_name = "employee_attendance_data.csv"

departments = {
    "IT": ["Software Developer", "Tester", "IT Support"],
    "HR": ["Recruiter", "HR Executive"],
    "Sales": ["Sales Executive", "Account Manager"],
    "Finance": ["Accountant", "Financial Analyst"],
    "Operations": ["Coordinator", "Operations Executive"]
}

headings = [
    "EmployeeID", "Department", "Role", "TenureYears",
    "ExperienceYears", "Satisfaction", "Overtime",
    "AttendanceDays", "WorkingDays", "Attrition"
]

# Create the CSV file automatically every time the notebook starts.
file = open(file_name, "w", newline="", encoding="utf-8")
writer = csv.writer(file)
writer.writerow(headings)

for i in range(1, 61):
    department_names = list(departments.keys())
    department = department_names[(i * 3 + i // 7) % len(department_names)]
    role_options = departments[department]
    role = role_options[i % len(role_options)]

    tenure = round(0.5 + ((i * 7) % 90) / 10, 1)
    experience = round(tenure + ((i * 3) % 50) / 10, 1)
    satisfaction = 1 + ((i * 2 + i // 4) % 5)

    if (i * 5) % 9 < 4:
        overtime = "Yes"
    else:
        overtime = "No"

    working_days = 22
    attendance_days = working_days - ((i * 7) % 6)

    leave_score = i * 11 + i // 3 + satisfaction
    if (satisfaction <= 2 and overtime == "Yes" and tenure < 5) or leave_score % 13 in (0, 1):
        attrition = "Yes"
    else:
        attrition = "No"

    writer.writerow([
        "E" + str(i).zfill(3), department, role, tenure,
        experience, satisfaction, overtime, attendance_days,
        working_days, attrition
    ])

file.close()
print("CSV file created successfully:", file_name)
print("Total employee records written:", 60)


CSV file created successfully: employee_attendance_data.csv
Total employee records written: 60


## 2. Read data from the CSV file

The project now reads the file using `csv.DictReader`. Each employee is stored as a dictionary, and all employees are stored in a list.

In [2]:
employees = []

file = open(file_name, "r", newline="", encoding="utf-8")
reader = csv.DictReader(file)

for row in reader:
    employee = {
        "EmployeeID": row["EmployeeID"],
        "Department": row["Department"],
        "Role": row["Role"],
        "TenureYears": float(row["TenureYears"]),
        "ExperienceYears": float(row["ExperienceYears"]),
        "Satisfaction": int(row["Satisfaction"]),
        "Overtime": row["Overtime"],
        "AttendanceDays": int(row["AttendanceDays"]),
        "WorkingDays": int(row["WorkingDays"]),
        "Attrition": row["Attrition"]
    }
    employees.append(employee)

file.close()

print("Records successfully read from CSV:", len(employees))


Records successfully read from CSV: 60


## 3. Table display function

This function prints data in aligned columns.

In [3]:
def show_table(title, headings, data):
    print("\n" + title)
    print("=" * 105)

    # Find a suitable width for every column.
    widths = []
    for column_number in range(len(headings)):
        width = len(str(headings[column_number]))
        for row in data:
            if len(str(row[column_number])) > width:
                width = len(str(row[column_number]))
        widths.append(width + 2)

    # Print the heading row.
    heading_line = ""
    for column_number in range(len(headings)):
        heading_line += str(headings[column_number]).ljust(widths[column_number]) + "|"
    print(heading_line)

    # Print a separator line.
    separator = ""
    for width in widths:
        separator += "-" * width + "+"
    print(separator)

    # Print each data row.
    for row in data:
        row_line = ""
        for column_number in range(len(headings)):
            row_line += str(row[column_number]).ljust(widths[column_number]) + "|"
        print(row_line)

    print("=" * 105)


def percentage(part, total):
    if total == 0:
        return 0
    return (part / total) * 100


## 4. View employee data in proper table format

The complete CSV dataset is displayed below as an aligned table.

In [4]:
employee_table = []

for employee in employees:
    employee_table.append([
        employee["EmployeeID"],
        employee["Department"],
        employee["Role"],
        employee["TenureYears"],
        employee["ExperienceYears"],
        employee["Satisfaction"],
        employee["Overtime"],
        employee["AttendanceDays"],
        employee["WorkingDays"],
        employee["Attrition"]
    ])

show_table("EMPLOYEE ATTENDANCE DATA", headings, employee_table)



EMPLOYEE ATTENDANCE DATA
EmployeeID  |Department  |Role                  |TenureYears  |ExperienceYears  |Satisfaction  |Overtime  |AttendanceDays  |WorkingDays  |Attrition  |
------------+------------+----------------------+-------------+-----------------+--------------+----------+----------------+-------------+-----------+
E001        |Finance     |Financial Analyst     |1.2          |1.5              |3             |No        |21              |22           |Yes        |
E002        |HR          |Recruiter             |1.9          |2.5              |5             |Yes       |20              |22           |Yes        |
E003        |Operations  |Operations Executive  |2.6          |3.5              |2             |No        |19              |22           |No         |
E004        |Sales       |Sales Executive       |3.3          |4.5              |5             |Yes       |18              |22           |No         |
E005        |IT          |IT Support            |4.0          |5.5  

## 5. Overall workforce summary

In [5]:
total_employees = len(employees)
total_leavers = 0
total_attendance = 0
total_working_days = 0

for employee in employees:
    if employee["Attrition"] == "Yes":
        total_leavers += 1
    total_attendance += employee["AttendanceDays"]
    total_working_days += employee["WorkingDays"]

summary_data = [
    ["Total employees", total_employees],
    ["Employees who left", total_leavers],
    ["Employees who stayed", total_employees - total_leavers],
    ["Overall attrition rate", str(round(percentage(total_leavers, total_employees), 2)) + "%"],
    ["Overall attendance rate", str(round(percentage(total_attendance, total_working_days), 2)) + "%"]
]

show_table("WORKFORCE SUMMARY", ["Metric", "Value"], summary_data)



WORKFORCE SUMMARY
Metric                   |Value   |
-------------------------+--------+
Total employees          |60      |
Employees who left       |14      |
Employees who stayed     |46      |
Overall attrition rate   |23.33%  |
Overall attendance rate  |88.64%  |


## 6. Department-wise employee distribution and attrition

In [6]:
department_counts = {}
department_leavers = {}

for employee in employees:
    department = employee["Department"]

    if department not in department_counts:
        department_counts[department] = 0
        department_leavers[department] = 0

    department_counts[department] += 1

    if employee["Attrition"] == "Yes":
        department_leavers[department] += 1

department_table = []
for department in department_counts:
    count = department_counts[department]
    left = department_leavers[department]
    rate = round(percentage(left, count), 2)
    department_table.append([department, count, left, count - left, str(rate) + "%"])

show_table(
    "DEPARTMENT-WISE ANALYSIS",
    ["Department", "Employees", "Left", "Stayed", "Attrition Rate"],
    department_table
)



DEPARTMENT-WISE ANALYSIS
Department  |Employees  |Left  |Stayed  |Attrition Rate  |
------------+-----------+------+--------+----------------+
Finance     |12         |2     |10      |16.67%          |
HR          |11         |4     |7       |36.36%          |
Operations  |12         |3     |9       |25.0%           |
Sales       |13         |2     |11      |15.38%          |
IT          |12         |3     |9       |25.0%           |


## 7. Role-wise attrition analysis

In [7]:
role_counts = {}
role_leavers = {}

for employee in employees:
    role = employee["Role"]

    if role not in role_counts:
        role_counts[role] = 0
        role_leavers[role] = 0

    role_counts[role] += 1

    if employee["Attrition"] == "Yes":
        role_leavers[role] += 1

role_table = []
for role in role_counts:
    count = role_counts[role]
    left = role_leavers[role]
    rate = round(percentage(left, count), 2)
    role_table.append([role, count, left, count - left, str(rate) + "%"])

show_table(
    "ROLE-WISE ATTRITION",
    ["Role", "Employees", "Left", "Stayed", "Attrition Rate"],
    role_table
)



ROLE-WISE ATTRITION
Role                  |Employees  |Left  |Stayed  |Attrition Rate  |
----------------------+-----------+------+--------+----------------+
Financial Analyst     |6          |2     |4       |33.33%          |
Recruiter             |6          |3     |3       |50.0%           |
Operations Executive  |6          |2     |4       |33.33%          |
Sales Executive       |6          |1     |5       |16.67%          |
IT Support            |5          |1     |4       |20.0%           |
Accountant            |6          |0     |6       |0.0%            |
Account Manager       |7          |1     |6       |14.29%          |
Tester                |4          |2     |2       |50.0%           |
Coordinator           |6          |1     |5       |16.67%          |
HR Executive          |5          |1     |4       |20.0%           |
Software Developer    |3          |0     |3       |0.0%            |


## 8. Attrition by employee tenure

In [8]:
tenure_groups = {
    "0 to 2 years": [0, 2],
    "Above 2 to 5 years": [2, 5],
    "Above 5 years": [5, 1000]
}

tenure_counts = {}
tenure_leavers = {}

for group in tenure_groups:
    tenure_counts[group] = 0
    tenure_leavers[group] = 0

for employee in employees:
    tenure = employee["TenureYears"]

    for group in tenure_groups:
        limits = tenure_groups[group]
        if tenure >= limits[0] and tenure <= limits[1]:
            tenure_counts[group] += 1
            if employee["Attrition"] == "Yes":
                tenure_leavers[group] += 1
            break

tenure_table = []
for group in tenure_groups:
    count = tenure_counts[group]
    left = tenure_leavers[group]
    rate = round(percentage(left, count), 2)
    tenure_table.append([group, count, left, count - left, str(rate) + "%"])

show_table(
    "TENURE-WISE ATTRITION",
    ["Tenure Group", "Employees", "Left", "Stayed", "Attrition Rate"],
    tenure_table
)



TENURE-WISE ATTRITION
Tenure Group        |Employees  |Left  |Stayed  |Attrition Rate  |
--------------------+-----------+------+--------+----------------+
0 to 2 years        |11         |4     |7       |36.36%          |
Above 2 to 5 years  |22         |9     |13      |40.91%          |
Above 5 years       |27         |1     |26      |3.7%            |


## 9. Satisfaction comparison: employees who left and stayed

In [9]:
satisfaction_totals = {"Yes": 0, "No": 0}
satisfaction_counts = {"Yes": 0, "No": 0}

for employee in employees:
    status = employee["Attrition"]
    satisfaction_totals[status] += employee["Satisfaction"]
    satisfaction_counts[status] += 1

satisfaction_table = []
for status in ["Yes", "No"]:
    if satisfaction_counts[status] > 0:
        average = satisfaction_totals[status] / satisfaction_counts[status]
        if status == "Yes":
            group_name = "Employees who left"
        else:
            group_name = "Employees who stayed"
        satisfaction_table.append([group_name, satisfaction_counts[status], round(average, 2)])

show_table(
    "AVERAGE SATISFACTION",
    ["Employee Group", "Employees", "Average Satisfaction (out of 5)"],
    satisfaction_table
)



AVERAGE SATISFACTION
Employee Group        |Employees  |Average Satisfaction (out of 5)  |
----------------------+-----------+---------------------------------+
Employees who left    |14         |2.5                              |
Employees who stayed  |46         |3.15                             |


## 10. Overtime and attendance analysis

In [10]:
overtime_counts = {"Yes": 0, "No": 0}
overtime_leavers = {"Yes": 0, "No": 0}
attendance_totals = {"Yes": 0, "No": 0}
attendance_counts = {"Yes": 0, "No": 0}

for employee in employees:
    overtime = employee["Overtime"]
    status = employee["Attrition"]

    overtime_counts[overtime] += 1
    if status == "Yes":
        overtime_leavers[overtime] += 1

    attendance_totals[status] += employee["AttendanceDays"]
    attendance_counts[status] += 1

overtime_table = []
for overtime in ["Yes", "No"]:
    count = overtime_counts[overtime]
    left = overtime_leavers[overtime]
    rate = round(percentage(left, count), 2)
    overtime_table.append([overtime, count, left, str(rate) + "%"])

show_table(
    "OVERTIME AND ATTRITION",
    ["Overtime", "Employees", "Left", "Attrition Rate"],
    overtime_table
)

attendance_table = []
for status in ["Yes", "No"]:
    if attendance_counts[status] > 0:
        average = attendance_totals[status] / attendance_counts[status]
        if status == "Yes":
            group_name = "Employees who left"
        else:
            group_name = "Employees who stayed"
        attendance_table.append([group_name, attendance_counts[status], round(average, 2)])

show_table(
    "AVERAGE ATTENDANCE",
    ["Employee Group", "Employees", "Average Attendance Days"],
    attendance_table
)



OVERTIME AND ATTRITION
Overtime  |Employees  |Left  |Attrition Rate  |
----------+-----------+------+----------------+
Yes       |27         |9     |33.33%          |
No        |33         |5     |15.15%          |

AVERAGE ATTENDANCE
Employee Group        |Employees  |Average Attendance Days  |
----------------------+-----------+-------------------------+
Employees who left    |14         |19.43                    |
Employees who stayed  |46         |19.52                    |


##PROJECT CONCLUSION

The project analysed employee attendance, departments, roles,
tenure, satisfaction, overtime and attrition using CSV data.
The results show group-level patterns for further HR investigation.
They do not prove why an employee leaves and must not be used
to judge or predict the behaviour of any individual employee.

Employee Attendance Analysis completed successfully.

##QUESTIONS HR SHOULD INVESTIGATE
                                                        
1    Are workload, overtime and scheduling contributing to low satisfaction?    
2    Are employees receiving enough training and career-development support?    
3    Do exit interviews identify common reasons for leaving?                    
4    Are attendance patterns affected by leave policies or shift arrangements?  
5    What additional information is needed before HR takes action?

##POSSIBLE RETENTION INITIATIVES
                                                
1    Review overtime levels and workload distribution.          
2    Offer career-development and training opportunities.       
3    Conduct confidential employee feedback sessions.           
4    Review attendance and leave policies.                      
5    Compare attrition rates before and after each initiative.

## Dataset information and download

- The notebook creates `employee_attendance_data.csv` automatically at the beginning.
- The CSV contains 60 synthetic sample employee records for academic practice.
- The notebook reads all analysis data from that CSV file.
- No manual file upload is needed.

####Project made by :-
#####Name - Aarnav Yadav
#####College ID - 2026B15410173